# Auditar un plan factible frente a una cota certificada

- **Level:** 2, Fundamentos
- **Prerequisites:** 01-problem-and-published-data
- **Reading time:** about 12 minutes

## Objetivos

Usar el lector de manifiestos de PhaseFlow para comparar resultados CPIT factibles con la cota certificada y comprobar invariantes básicos de orden y brecha.

## Preparación

Environment pins come from `notebooks/requirements.txt`. The product implementation remains in `data-pipeline/`; this notebook imports its manifest helper.

```bash
pip install oreblocks==0.6.3 numpy==2.5.1 scipy==1.18.1 matplotlib==3.11.2
```

In [1]:
# 02 bound and feasible plan
import importlib.metadata as metadata
import json
from pathlib import Path

for package in ("oreblocks", "numpy", "scipy", "matplotlib"):
    print(package, metadata.version(package))

oreblocks 0.6.3
numpy 2.5.1
scipy 1.18.1
matplotlib 3.11.2


## Datos

El cuaderno lee el manifiesto comprometido. No reconstruye bloques de MineLib ni infiere planes por bloque que no están disponibles.

In [2]:
# 02 bound and feasible plan
manifest_path = Path("../../../data/derived/manifests/newman1-published.json")
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
print("case:", manifest.get("case_id", manifest.get("case")))
print("scenario:", manifest.get("scenario"))
print("trace bytes:", manifest.get("trace_bytes"))

case: newman1-published
scenario: {'periods': 6, 'discount_rate': 0.08, 'period_one_undiscounted': True, 'n_resources': 2, 'declared': False, 'capacity_fraction': [2.1367, 2.1859], 'limit_per_period': [2000000.0, 1100000.0], 'resource_names': ['mining', 'processing']}
trace bytes: None


## Método

La función best_comparable del producto selecciona solo resultados elegibles para la envolvente CPIT. Las comprobaciones independientes comparan el valor del plan con la cota declarada y calculan la brecha relativa.

In [3]:
# 02 bound and feasible plan
import sys
sys.path.insert(0, str(Path("../../../data-pipeline").resolve()))
from types import SimpleNamespace
from pipeline.core.manifest import best_comparable
results = [SimpleNamespace(**row) for row in manifest["scoreboard"]]
best = best_comparable(results)
print("comparable result:", best.method, "NPV:", round(best.npv))
assert best.rung in {"classical", "sota", "learned"}

comparable result: sliding-window NPV: 24142200


## Resultados

Esta comparación depende de la formulación: los resultados PCPSP y los resultados enteros externos atribuidos deben conservar etiquetas separadas.

In [4]:
# 02 bound and feasible plan
assert manifest_path.is_file()
assert manifest.get("case_id", manifest.get("case")) is not None
print("manifest provenance check: passed")

manifest provenance check: passed


## Resumen

Se inspeccionó evidencia trazable; no se extrapoló desde un caso.

## Referencias

- MineLib, benchmark instances and published results: https://minelib.org/v1/Results.xhtml
- Chicoisne et al. (2012), https://doi.org/10.1287/opre.1120.1050
- PhaseFlow case dossier: `docs/use-cases/01_newman1-published.md`